In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
%run /Workspace/Users/sushantbhardwaj15@gmail.com/FMCG_Analytics/setup_utils_config/Utilities

## DQ CHECKS

In [0]:
# ── DQ rules per table ────────────────────────────────────────────────────────
# Each entry: (filter condition, fail reason message)

DQ_RULES = {
    "customers": (
        F.col("customer_id").isNotNull() & F.col("customer_id").rlike("^[0-9]+$") & F.col("customer_name").isNotNull(),
        "customer_id is null or or customer_name is null",
    ),
    "products": (
        F.col("product_id").isNotNull() & F.col("product_id").rlike("^[0-9]+$") & F.col("product_name").isNotNull(),
        "product_id or product_name is null",
    ),
    "gross_price": (
        F.col("product_id").isNotNull() & (F.col("product_id").rlike("^[0-9]+$")) & (F.col("gross_price").rlike("^[0-9]+$")),
        "product_id null or gross_price <= 0 ",
    ),
    "orders": (
        F.col("order_id").isNotNull() & ((F.col("order_qty").isNotNull())),
        "order_id null or quantity is null",
    ),
}

In [0]:
PRE_QUARANTINE_CASTS = {
    "products": {
        "product_id": "string"
    }
}
def apply_pre_quarantine_casts(df, table_name):
    """
    Apply table-specific column casts before quarantine.
    """
    casts = PRE_QUARANTINE_CASTS.get(table_name, {})

    for col_name, data_type in casts.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(data_type))

    return df

In [0]:
# ── CUSTOMERS, PRODUCTS, GROSS PRICE (full tables from bronze) ────────────────
good_dfs = {}
bad_dfs ={}
for table_name in ["customers", "products", "gross_price"]:
    print(f"\n DQ checks for {table_name}")

    df = spark.table(f"{catalog}.bronze.{table_name}")

    condition, fail_reason = DQ_RULES[table_name]
    good, bad = split_good_bad(df, condition, fail_reason)
    bad = apply_pre_quarantine_casts(bad, table_name)

    save_quarantine(bad, f"{catalog}.bronze.{table_name}",f"{table_name}")

    good = good.withColumn(
        "_silver_loaded_at", F.current_timestamp()
    )
    good_dfs[table_name] = good
    bad_dfs[table_name] =bad


## Customers Processing

In [0]:
df_bronze = good_dfs["customers"]
display(df_bronze)

In [0]:
df_duplicate = df_bronze.groupBy(F.col("customer_id")).agg(F.count("*").alias("count"))
display(df_duplicate.filter(F.col("count") > 1))

In [0]:
df_filtered = df_bronze.filter(F.col("customer_id")=="789321")
display(df_filtered)

In [0]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('Rows after duplicates dropped: ', df_silver.count())

In [0]:
# check those values
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))
display(df_silver) 

In [0]:
# check those values
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
display(df_silver.select(F.col("city")))

In [0]:
city_mapping = {
    "Bengaluru": "Bengaluru",
    "Bengalore": "Bengaluru",
    "Bengaluruu": "Bengaluru",
    "Hyderabad": "Hyderabad",
    "Hyderabadd": "Hyderabad",
    "Hyderbad": "Hyderabad",
    "New Delhi": "New Delhi",
    "NewDelhi": "New Delhi",
    "NewDelhee": "New Delhi",
    "NewDheli": "New Delhi"
}

mapping_expr = F.create_map(
    *[F.lit(x) for x in sum(city_mapping.items(), ())]
)

df_silver = df_silver.withColumn(
    "city_standardized",
    mapping_expr[F.col("city")]
).drop("city")

In [0]:
df_silver.select(F.col("city_standardized")).distinct().show()

In [0]:
df_silver = df_silver.withColumnRenamed("city_standardized","city")

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)

# sanity check

df_silver.select('customer_name').distinct().show()

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
df_join = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

In [0]:
# Sanity Checks

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_join.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
df_join = df_join.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_join.printSchema())

In [0]:
df_join = (
    df_join
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_join.limit(5))

In [0]:
#records count check
df_join.count()

In [0]:
df_join.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.customers")

In [0]:
write_audit("silver", "customers", df_join.count(), "SUCCESS",
f"quarantined={bad_dfs["customers"].count()}")

## Product Processing

In [0]:
df_bronze = good_dfs["products"]
display(df_bronze)

In [0]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['product_id'])
print('Rows after duplicates dropped: ', df_silver.count())

In [0]:
df_silver.select('category').distinct().show()

In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(), None)
     .otherwise(F.initcap("category"))
)

In [0]:
df_silver.select('category').distinct().show()

In [0]:
# Replace 'protien' → 'protein' in both product_name and category
df_silver = (
    df_silver
    .withColumn(
        "product_name",
        F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein")
    )
    .withColumn(
        "category",
        F.regexp_replace(F.col("category"), "(?i)Protien", "Protein")
    )
)


In [0]:
display(df_silver.limit(5))

In [0]:
%sql
DESCRIBE TABLE EXTENDED fmcg.bronze.customers;

## Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
### 1: Add division column
df_silver = (
    df_silver
    .withColumn(
        "division",
        F.when(F.col("category") == "Energy Bars",        "Nutrition Bars")
         .when(F.col("category") == "Protein Bars",       "Nutrition Bars")
         .when(F.col("category") == "Granola & Cereals",  "Breakfast Foods")
         .when(F.col("category") == "Recovery Dairy",     "Dairy & Recovery")
         .when(F.col("category") == "Healthy Snacks",     "Healthy Snacks")
         .when(F.col("category") == "Electrolyte Mix",    "Hydration & Electrolytes")
         .otherwise("Other")
    )
)


### 2: Variant column
df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)


### 3: Create new column: product_code  

# Invalid product_ids are replaced with a fallback value to avoid losing fact records and ensure downstream joins remain consistent

df_silver = (
    df_silver
    # 1. Generate deterministic product_code from product_name
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    # 2. Clean product_id: keep only numeric IDs, else set to 999999
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
    # 3. Rename product_name → product
    .withColumnRenamed("product_name", "product")
)

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "_source_file", "_ingested_at", "_load_date","_silver_loaded_at")

In [0]:
display(df_silver)

In [0]:
df_silver.count()

In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.products")

In [0]:
write_audit("silver", "products", df_silver.count(), "SUCCESS",
f"quarantined={bad_dfs["products"].count()}")

## Gross Price Processing

In [0]:
df_bronze = good_dfs["gross_price"]
display(df_bronze)

In [0]:
df_bronze.select('month').distinct().show()

In [0]:

# 1️. Parse `month` from multiple possible formats
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = df_bronze.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)

In [0]:
df_silver.select('month').distinct().show()

In [0]:
# We enrich the silver dataset by performing an inner join with the products table to fetch the correct product_code for each product_id.

df_products = spark.table("fmcg.silver.products") 
df_joined = df_silver.join(df_products.select("product_id", "product_code"), on="product_id", how="inner")

In [0]:
display(df_joined)

In [0]:
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "_source_file","_ingested_at", "_load_date", "_silver_loaded_at")

df_joined.show(5)

In [0]:
df_joined.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true")\
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.gross_price")

In [0]:
write_audit("silver", "gross_price", df_joined.count(), "SUCCESS",
f"quarantined={bad_dfs["gross_price"].count()}")